# H2分子激发态VMC方法对比：QNN vs FFNN

本notebook对比两种不同的神经网络方法用于H2分子激发态的变分蒙特卡洛(VMC)计算：
- **QNN (Quantum Neural Network)**: 使用PennyLane实现的量子神经网络
- **FFNN (Feed-Forward Neural Network)**: 传统的前馈神经网络

两种方法都使用NetKet框架进行VMC计算，计算第一激发态和第二激发态，惩罚系数统一设置为 [3.0, 3.0]。

## 1. 环境设置和分子定义

In [ ]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
import pickle
from pyscf import gto, scf, fci
import netket.experimental as nkx
import jax
import jax.numpy as jnp
import pennylane as qml
from flax import nnx
from functools import partial, reduce
import sys
sys.path.append('激发态能量/Netket_excited_state')
import vmc_ex

jax.config.update("jax_enable_x64", True)

In [ ]:
bond_length = 1.4
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

mol = gto.M(atom=geometry, basis='STO-3G')

mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"Hartree-Fock能量: {E_hf:.8f} Ha")

cisolver = fci.FCI(mf)
cisolver.nroots = 3
E_fcis, fcivec = cisolver.kernel()
print(f"FCI基态能量: {E_fcis[0]:.8f} Ha")
print(f"\nFCI所有能级:")
for i, e in enumerate(E_fcis):
    print(f"  E{i} = {e:.8f} Ha")

ha = nkx.operator.from_pyscf_molecule(mol)

In [ ]:
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

g = nk.graph.Graph(edges=[(0,1),(2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)
print(f"采样器图边数: {g.n_edges}")

## 2. QNN方法实现

### 2.1 定义量子神经网络

In [ ]:
def quantum_neural_network(x, params, n_qubits, n_layers):
    x = jnp.atleast_2d(x)
    if x.shape[-1] != n_qubits:
        raise ValueError(f"输入特征维度需为{n_qubits}，当前为{x.shape[-1]}")
    
    for i in range(n_qubits):
        qml.RX(x[:, i] * jnp.pi, wires=i)
    
    for layer in range(n_layers):
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
        qml.Barrier(wires=range(n_qubits))
        for i in range(n_qubits):
            qml.RX(params[layer, 2*i], wires=i)
            qml.RZ(params[layer, 2*i+1], wires=i)
    
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [ ]:
def qnn_circuit(n_qubits: int, n_layers: int):
    dev = qml.device('default.qubit', wires=n_qubits)
    pqc_node = qml.QNode(func=quantum_neural_network, device=dev, interface='jax')
    qnn_node_circuit = partial(pqc_node, n_qubits=n_qubits, n_layers=n_layers)
    return qnn_node_circuit

In [ ]:
class QNNLinear(nnx.Module):
    def __init__(self, rngs: nnx.Rngs, n_qubits: int, n_layer: int):
        key = rngs.params()
        self.n_qubits, self.n_layer = n_qubits, n_layer
        self.qnn_params = nnx.Param(jax.random.normal(key, (self.n_layer, 2*self.n_qubits), dtype=jnp.float32))
        self.qnn_layer = partial(qnn_circuit, n_qubits=self.n_qubits, n_layers=self.n_layer)()
        self.Linear = nnx.Linear(in_features=self.n_qubits, out_features=self.n_qubits, use_bias=False, rngs=rngs)
        
    def __call__(self, s: np.array):
        s = jnp.array(s, dtype=jnp.float32)
        qnn_output = self.qnn_layer(x=s, params=self.qnn_params)
        qnn_output = jnp.stack(qnn_output, axis=1)
        qnn_output = jnp.array(qnn_output, dtype=jnp.float32)
        y = self.Linear(qnn_output)
        y = nnx.relu(y)
        return jnp.sum(y, axis=-1)

class QNN_Hybrid(nnx.Module):
    def __init__(self, rngs_real: nnx.Rngs, rngs_imag: nnx.Rngs, n_qubits: int, n_layer: int):
        self.qnn_real = QNNLinear(rngs_real, n_qubits, n_layer)
        self.qnn_imag = QNNLinear(rngs_imag, n_qubits, n_layer)
        
    def __call__(self, s: np.array):
        real = self.qnn_real(s)
        imag = self.qnn_imag(s)
        return real + 1j*imag

## 3. FFNN方法实现

### 3.1 定义前馈神经网络

In [ ]:
class FFN(nnx.Module):
    def __init__(self, N: int, alpha: int = 1, *, rngs: nnx.Rngs):
        self.alpha = alpha
        self.linear = nnx.Linear(in_features=N, out_features=alpha * N, rngs=rngs, param_dtype=complex)
    
    def __call__(self, x: jax.Array):
        y = self.linear(x)
        y = nnx.relu(y)
        return jnp.sum(y, axis=-1)

## 4. 基态计算

### 4.1 QNN基态计算

In [ ]:
n_qubits = 4
n_layer = 2

qnn_model_gs = QNN_Hybrid(
    rngs_real=nnx.Rngs(params=0),
    rngs_imag=nnx.Rngs(params=1),
    n_qubits=n_qubits,
    n_layer=n_layer
)

qnn_vstate_gs = nk.vqs.MCState(sampler, qnn_model_gs, n_samples=1024)

opt_qnn_gs = nk.optimizer.Sgd(learning_rate=0.05)
sr_qnn_gs = nk.optimizer.SR(diag_shift=0.01)

qnn_gs_driver = nk.driver.VMC(ha, opt_qnn_gs, variational_state=qnn_vstate_gs, preconditioner=sr_qnn_gs)

qnn_gs_exp_name = "Data_QNN_FFNN/qnn_ground_state"
print("开始QNN基态训练...")

In [ ]:
qnn_gs_driver.run(300, out=qnn_gs_exp_name)
print("QNN基态训练完成！")

data_qnn_gs = json.load(open(qnn_gs_exp_name + '.log'))
qnn_gs_energy = data_qnn_gs["Energy"]["Mean"]["real"]
qnn_gs_final = reduce(lambda x, y: x if y is None else y, qnn_gs_energy)
print(f"QNN基态能量: {qnn_gs_final:.8f} Ha")
print(f"与FCI误差: {abs(qnn_gs_final - E_fcis[0]):.8f} Ha")

In [ ]:
import os
os.makedirs('Data_QNN_FFNN', exist_ok=True)

qnn_gs_params = qnn_vstate_gs.parameters
with open('Data_QNN_FFNN/qnn_gs.json', 'wb') as f:
    pickle.dump(qnn_gs_params, f)
print("QNN基态参数已保存")

### 4.2 FFNN基态计算

In [ ]:
N = 4
ffnn_model_gs = FFN(N=N, alpha=1, rngs=nnx.Rngs(2))

ffnn_vstate_gs = nk.vqs.MCState(sampler, ffnn_model_gs, n_discard_per_chain=10, n_samples=1024)

opt_ffnn_gs = nk.optimizer.Sgd(learning_rate=0.05)
sr_ffnn_gs = nk.optimizer.SR(diag_shift=0.01)

ffnn_gs_driver = nk.driver.VMC(ha, opt_ffnn_gs, variational_state=ffnn_vstate_gs, preconditioner=sr_ffnn_gs)

ffnn_gs_exp_name = "Data_QNN_FFNN/ffnn_ground_state"
print("开始FFNN基态训练...")

In [ ]:
ffnn_gs_driver.run(300, out=ffnn_gs_exp_name)
print("FFNN基态训练完成！")

data_ffnn_gs = json.load(open(ffnn_gs_exp_name + '.log'))
ffnn_gs_energy = data_ffnn_gs["Energy"]["Mean"]["real"]
ffnn_gs_final = reduce(lambda x, y: x if y is None else y, ffnn_gs_energy)
print(f"FFNN基态能量: {ffnn_gs_final:.8f} Ha")
print(f"与FCI误差: {abs(ffnn_gs_final - E_fcis[0]):.8f} Ha")

In [ ]:
ffnn_gs_params = ffnn_vstate_gs.parameters
with open('Data_QNN_FFNN/ffnn_gs.json', 'wb') as f:
    pickle.dump(ffnn_gs_params, f)
print("FFNN基态参数已保存")

## 5. 第一激发态计算（惩罚系数 [3.0]）

### 5.1 QNN第一激发态计算

In [ ]:
with open('Data_QNN_FFNN/qnn_gs.json', 'rb') as f:
    qnn_gs_params = pickle.load(f)

qnn_model_gs_loaded = QNN_Hybrid(
    rngs_real=nnx.Rngs(params=0),
    rngs_imag=nnx.Rngs(params=1),
    n_qubits=n_qubits,
    n_layer=n_layer
)
qnn_vstate_gs_loaded = nk.vqs.MCState(sampler, qnn_model_gs_loaded, n_samples=1024)
qnn_vstate_gs_loaded.parameters = qnn_gs_params
print("QNN基态参数已加载")

In [ ]:
qnn_model_ex1 = QNN_Hybrid(
    rngs_real=nnx.Rngs(params=10),
    rngs_imag=nnx.Rngs(params=11),
    n_qubits=n_qubits,
    n_layer=n_layer
)
qnn_vstate_ex1 = nk.vqs.MCState(sampler, qnn_model_ex1, n_samples=1024)

opt_qnn_ex1 = nk.optimizer.Sgd(learning_rate=0.05)
sr_qnn_ex1 = nk.optimizer.SR(diag_shift=0.01)

shift_list_ex1 = [3.0]
state_list_qnn_ex1 = [qnn_vstate_gs_loaded]

qnn_ex1_driver = vmc_ex.VMC_ex(
    hamiltonian=ha,
    optimizer=opt_qnn_ex1,
    variational_state=qnn_vstate_ex1,
    preconditioner=sr_qnn_ex1,
    state_list=state_list_qnn_ex1,
    shift_list=shift_list_ex1
)

qnn_ex1_exp_name = "Data_QNN_FFNN/qnn_excited_state_1"
print("开始QNN第一激发态训练...")

In [ ]:
qnn_ex1_driver.run(500, out=qnn_ex1_exp_name)
print("QNN第一激发态训练完成！")

data_qnn_ex1 = json.load(open(qnn_ex1_exp_name + '.log'))
qnn_ex1_energy = data_qnn_ex1["Energy"]["Mean"]["real"]
qnn_ex1_final = reduce(lambda x, y: x if y is None else y, qnn_ex1_energy)
print(f"\nQNN第一激发态能量: {qnn_ex1_final:.8f} Ha | 精确解: {E_fcis[1]:.8f} Ha")
print(f"激发能: {qnn_ex1_final - qnn_gs_final:.8f} Ha")
print(f"与FCI第一激发态误差: {abs(qnn_ex1_final - E_fcis[1]):.8f} Ha")

In [ ]:
qnn_ex1_params = qnn_vstate_ex1.parameters
with open('Data_QNN_FFNN/qnn_ex1.json', 'wb') as f:
    pickle.dump(qnn_ex1_params, f)
print("QNN第一激发态参数已保存")

### 5.2 FFNN第一激发态计算

In [ ]:
with open('Data_QNN_FFNN/ffnn_gs.json', 'rb') as f:
    ffnn_gs_params = pickle.load(f)

ffnn_model_gs_loaded = FFN(N=N, alpha=1, rngs=nnx.Rngs(2))
ffnn_vstate_gs_loaded = nk.vqs.MCState(sampler, ffnn_model_gs_loaded, n_discard_per_chain=10, n_samples=1024)
ffnn_vstate_gs_loaded.parameters = ffnn_gs_params
print("FFNN基态参数已加载")

In [ ]:
ffnn_model_ex1 = FFN(N=N, alpha=1, rngs=nnx.Rngs(20))
ffnn_vstate_ex1 = nk.vqs.MCState(sampler, ffnn_model_ex1, n_discard_per_chain=10, n_samples=1024)

opt_ffnn_ex1 = nk.optimizer.Sgd(learning_rate=0.05)
sr_ffnn_ex1 = nk.optimizer.SR(diag_shift=0.01)

shift_list_ex1 = [3.0]
state_list_ffnn_ex1 = [ffnn_vstate_gs_loaded]

ffnn_ex1_driver = vmc_ex.VMC_ex(
    hamiltonian=ha,
    optimizer=opt_ffnn_ex1,
    variational_state=ffnn_vstate_ex1,
    preconditioner=sr_ffnn_ex1,
    state_list=state_list_ffnn_ex1,
    shift_list=shift_list_ex1
)

ffnn_ex1_exp_name = "Data_QNN_FFNN/ffnn_excited_state_1"
print("开始FFNN第一激发态训练...")

In [ ]:
ffnn_ex1_driver.run(500, out=ffnn_ex1_exp_name)
print("FFNN第一激发态训练完成！")

data_ffnn_ex1 = json.load(open(ffnn_ex1_exp_name + '.log'))
ffnn_ex1_energy = data_ffnn_ex1["Energy"]["Mean"]["real"]
ffnn_ex1_final = reduce(lambda x, y: x if y is None else y, ffnn_ex1_energy)
print(f"\nFFNN第一激发态能量: {ffnn_ex1_final:.8f} Ha | 精确解: {E_fcis[1]:.8f} Ha")
print(f"激发能: {ffnn_ex1_final - ffnn_gs_final:.8f} Ha")
print(f"与FCI第一激发态误差: {abs(ffnn_ex1_final - E_fcis[1]):.8f} Ha")

In [ ]:
ffnn_ex1_params = ffnn_vstate_ex1.parameters
with open('Data_QNN_FFNN/ffnn_ex1.json', 'wb') as f:
    pickle.dump(ffnn_ex1_params, f)
print("FFNN第一激发态参数已保存")

## 6. 第二激发态计算（惩罚系数 [3.0, 3.0]）

### 6.1 QNN第二激发态计算

In [ ]:
with open('Data_QNN_FFNN/qnn_ex1.json', 'rb') as f:
    qnn_ex1_params = pickle.load(f)

qnn_model_ex1_loaded = QNN_Hybrid(
    rngs_real=nnx.Rngs(params=10),
    rngs_imag=nnx.Rngs(params=11),
    n_qubits=n_qubits,
    n_layer=n_layer
)
qnn_vstate_ex1_loaded = nk.vqs.MCState(sampler, qnn_model_ex1_loaded, n_samples=1024)
qnn_vstate_ex1_loaded.parameters = qnn_ex1_params
print("QNN第一激发态参数已加载")

In [ ]:
qnn_model_ex2 = QNN_Hybrid(
    rngs_real=nnx.Rngs(params=20),
    rngs_imag=nnx.Rngs(params=21),
    n_qubits=n_qubits,
    n_layer=n_layer
)
qnn_vstate_ex2 = nk.vqs.MCState(sampler, qnn_model_ex2, n_samples=1024)

opt_qnn_ex2 = nk.optimizer.Sgd(learning_rate=0.05)
sr_qnn_ex2 = nk.optimizer.SR(diag_shift=0.01)

shift_list_ex2 = [3.0, 3.0]
state_list_qnn_ex2 = [qnn_vstate_gs_loaded, qnn_vstate_ex1_loaded]

qnn_ex2_driver = vmc_ex.VMC_ex(
    hamiltonian=ha,
    optimizer=opt_qnn_ex2,
    variational_state=qnn_vstate_ex2,
    preconditioner=sr_qnn_ex2,
    state_list=state_list_qnn_ex2,
    shift_list=shift_list_ex2
)

qnn_ex2_exp_name = "Data_QNN_FFNN/qnn_excited_state_2"
print("开始QNN第二激发态训练...")

In [ ]:
qnn_ex2_driver.run(500, out=qnn_ex2_exp_name)
print("QNN第二激发态训练完成！")

data_qnn_ex2 = json.load(open(qnn_ex2_exp_name + '.log'))
qnn_ex2_energy = data_qnn_ex2["Energy"]["Mean"]["real"]
qnn_ex2_final = reduce(lambda x, y: x if y is None else y, qnn_ex2_energy)
print(f"\nQNN第二激发态能量: {qnn_ex2_final:.8f} Ha | 精确解: {E_fcis[2]:.8f} Ha")
print(f"激发能: {qnn_ex2_final - qnn_gs_final:.8f} Ha")
print(f"与FCI第二激发态误差: {abs(qnn_ex2_final - E_fcis[2]):.8f} Ha")

In [ ]:
qnn_ex2_params = qnn_vstate_ex2.parameters
with open('Data_QNN_FFNN/qnn_ex2.json', 'wb') as f:
    pickle.dump(qnn_ex2_params, f)
print("QNN第二激发态参数已保存")

### 6.2 FFNN第二激发态计算

In [ ]:
with open('Data_QNN_FFNN/ffnn_ex1.json', 'rb') as f:
    ffnn_ex1_params = pickle.load(f)

ffnn_model_ex1_loaded = FFN(N=N, alpha=1, rngs=nnx.Rngs(20))
ffnn_vstate_ex1_loaded = nk.vqs.MCState(sampler, ffnn_model_ex1_loaded, n_discard_per_chain=10, n_samples=1024)
ffnn_vstate_ex1_loaded.parameters = ffnn_ex1_params
print("FFNN第一激发态参数已加载")

In [ ]:
ffnn_model_ex2 = FFN(N=N, alpha=1, rngs=nnx.Rngs(30))
ffnn_vstate_ex2 = nk.vqs.MCState(sampler, ffnn_model_ex2, n_discard_per_chain=10, n_samples=1024)

opt_ffnn_ex2 = nk.optimizer.Sgd(learning_rate=0.05)
sr_ffnn_ex2 = nk.optimizer.SR(diag_shift=0.01)

shift_list_ex2 = [3.0, 3.0]
state_list_ffnn_ex2 = [ffnn_vstate_gs_loaded, ffnn_vstate_ex1_loaded]

ffnn_ex2_driver = vmc_ex.VMC_ex(
    hamiltonian=ha,
    optimizer=opt_ffnn_ex2,
    variational_state=ffnn_vstate_ex2,
    preconditioner=sr_ffnn_ex2,
    state_list=state_list_ffnn_ex2,
    shift_list=shift_list_ex2
)

ffnn_ex2_exp_name = "Data_QNN_FFNN/ffnn_excited_state_2"
print("开始FFNN第二激发态训练...")

In [ ]:
ffnn_ex2_driver.run(500, out=ffnn_ex2_exp_name)
print("FFNN第二激发态训练完成！")

data_ffnn_ex2 = json.load(open(ffnn_ex2_exp_name + '.log'))
ffnn_ex2_energy = data_ffnn_ex2["Energy"]["Mean"]["real"]
ffnn_ex2_final = reduce(lambda x, y: x if y is None else y, ffnn_ex2_energy)
print(f"\nFFNN第二激发态能量: {ffnn_ex2_final:.8f} Ha | 精确解: {E_fcis[2]:.8f} Ha")
print(f"激发能: {ffnn_ex2_final - ffnn_gs_final:.8f} Ha")
print(f"与FCI第二激发态误差: {abs(ffnn_ex2_final - E_fcis[2]):.8f} Ha")

In [ ]:
ffnn_ex2_params = ffnn_vstate_ex2.parameters
with open('Data_QNN_FFNN/ffnn_ex2.json', 'wb') as f:
    pickle.dump(ffnn_ex2_params, f)
print("FFNN第二激发态参数已保存")

## 7. 结果对比分析

In [ ]:
def load_log_data(exp_name):
    with open(f"{exp_name}.log") as f:
        data = json.load(f)
    iters = data["Energy"]["iters"]
    energies = data["Energy"]["Mean"]["real"]
    variance = data["Energy"]["Variance"]
    return iters, energies, variance

qnn_gs_iters, qnn_gs_energies, qnn_gs_var = load_log_data(qnn_gs_exp_name)
ffnn_gs_iters, ffnn_gs_energies, ffnn_gs_var = load_log_data(ffnn_gs_exp_name)
qnn_ex1_iters, qnn_ex1_energies, qnn_ex1_var = load_log_data(qnn_ex1_exp_name)
ffnn_ex1_iters, ffnn_ex1_energies, ffnn_ex1_var = load_log_data(ffnn_ex1_exp_name)
qnn_ex2_iters, qnn_ex2_energies, qnn_ex2_var = load_log_data(qnn_ex2_exp_name)
ffnn_ex2_iters, ffnn_ex2_energies, ffnn_ex2_var = load_log_data(ffnn_ex2_exp_name)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].axhline(E_fcis[0], color="purple", linestyle="--", linewidth=2, label=f"FCI E0 = {E_fcis[0]:.6f} Ha")
axes[0, 0].plot(qnn_gs_iters, qnn_gs_energies, 'b-', linewidth=2, label="QNN")
axes[0, 0].plot(ffnn_gs_iters, ffnn_gs_energies, 'r-', linewidth=2, label="FFNN")
axes[0, 0].set_xlabel("Iteration", fontsize=12)
axes[0, 0].set_ylabel("Energy (Ha)", fontsize=12)
axes[0, 0].set_title("基态能量收敛", fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].axhline(E_fcis[1], color="purple", linestyle="--", linewidth=2, label=f"FCI E1 = {E_fcis[1]:.6f} Ha")
axes[0, 1].plot(qnn_ex1_iters, qnn_ex1_energies, 'b-', linewidth=2, label="QNN")
axes[0, 1].plot(ffnn_ex1_iters, ffnn_ex1_energies, 'r-', linewidth=2, label="FFNN")
axes[0, 1].set_xlabel("Iteration", fontsize=12)
axes[0, 1].set_ylabel("Energy (Ha)", fontsize=12)
axes[0, 1].set_title("第一激发态能量收敛 (惩罚系数=3.0)", fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].axhline(E_fcis[2], color="purple", linestyle="--", linewidth=2, label=f"FCI E2 = {E_fcis[2]:.6f} Ha")
axes[1, 0].plot(qnn_ex2_iters, qnn_ex2_energies, 'b-', linewidth=2, label="QNN")
axes[1, 0].plot(ffnn_ex2_iters, ffnn_ex2_energies, 'r-', linewidth=2, label="FFNN")
axes[1, 0].set_xlabel("Iteration", fontsize=12)
axes[1, 0].set_ylabel("Energy (Ha)", fontsize=12)
axes[1, 0].set_title("第二激发态能量收敛 (惩罚系数=[3.0, 3.0])", fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(qnn_gs_iters, qnn_gs_var, 'b-', linewidth=2, label="QNN 基态")
axes[1, 1].plot(ffnn_gs_iters, ffnn_gs_var, 'r-', linewidth=2, label="FFNN 基态")
axes[1, 1].plot(qnn_ex1_iters, qnn_ex1_var, 'b--', linewidth=2, label="QNN 第一激发态")
axes[1, 1].plot(ffnn_ex1_iters, ffnn_ex1_var, 'r--', linewidth=2, label="FFNN 第一激发态")
axes[1, 1].plot(qnn_ex2_iters, qnn_ex2_var, 'b:', linewidth=2, label="QNN 第二激发态")
axes[1, 1].plot(ffnn_ex2_iters, ffnn_ex2_var, 'r:', linewidth=2, label="FFNN 第二激发态")
axes[1, 1].set_xlabel("Iteration", fontsize=12)
axes[1, 1].set_ylabel("Local Energy Variance", fontsize=12)
axes[1, 1].set_title("能量方差收敛对比", fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=8, ncol=2)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
print("="*80)
print("H2分子激发态VMC计算结果对比")
print("惩罚系数: 第一激发态 [3.0], 第二激发态 [3.0, 3.0]")
print("="*80)
print()
print("参考值 (FCI):")
for i, e in enumerate(E_fcis):
    print(f"  E{i} = {e:.8f} Ha")
print()
print("QNN方法结果:")
print(f"  基态能量:       {qnn_gs_final:.8f} Ha, 误差: {abs(qnn_gs_final - E_fcis[0]):.8f} Ha")
print(f"  第一激发态能量: {qnn_ex1_final:.8f} Ha, 误差: {abs(qnn_ex1_final - E_fcis[1]):.8f} Ha")
print(f"  第二激发态能量: {qnn_ex2_final:.8f} Ha, 误差: {abs(qnn_ex2_final - E_fcis[2]):.8f} Ha")
print(f"  第一激发能:     {qnn_ex1_final - qnn_gs_final:.8f} Ha")
print(f"  第二激发能:     {qnn_ex2_final - qnn_gs_final:.8f} Ha")
print()
print("FFNN方法结果:")
print(f"  基态能量:       {ffnn_gs_final:.8f} Ha, 误差: {abs(ffnn_gs_final - E_fcis[0]):.8f} Ha")
print(f"  第一激发态能量: {ffnn_ex1_final:.8f} Ha, 误差: {abs(ffnn_ex1_final - E_fcis[1]):.8f} Ha")
print(f"  第二激发态能量: {ffnn_ex2_final:.8f} Ha, 误差: {abs(ffnn_ex2_final - E_fcis[2]):.8f} Ha")
print(f"  第一激发能:     {ffnn_ex1_final - ffnn_gs_final:.8f} Ha")
print(f"  第二激发能:     {ffnn_ex2_final - ffnn_gs_final:.8f} Ha")
print()
print("对比分析:")
print(f"  基态误差差异:       QNN比FFNN优 {abs(ffnn_gs_final - E_fcis[0]) - abs(qnn_gs_final - E_fcis[0]):.8f} Ha")
print(f"  第一激发态误差差异: QNN比FFNN优 {abs(ffnn_ex1_final - E_fcis[1]) - abs(qnn_ex1_final - E_fcis[1]):.8f} Ha")
print(f"  第二激发态误差差异: QNN比FFNN优 {abs(ffnn_ex2_final - E_fcis[2]) - abs(qnn_ex2_final - E_fcis[2]):.8f} Ha")
print("="*80)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

states = ['基态 (E0)', '第一激发态 (E1)', '第二激发态 (E2)']
qnn_energies = [qnn_gs_final, qnn_ex1_final, qnn_ex2_final]
ffnn_energies = [ffnn_gs_final, ffnn_ex1_final, ffnn_ex2_final]
fci_energies = list(E_fcis[:3])

x = np.arange(len(states))
width = 0.25

rects1 = ax.bar(x - width, qnn_energies, width, label='QNN', color='blue', alpha=0.7)
rects2 = ax.bar(x, ffnn_energies, width, label='FFNN', color='red', alpha=0.7)
rects3 = ax.bar(x + width, fci_energies, width, label='FCI', color='green', alpha=0.7)

ax.set_ylabel('Energy (Ha)', fontsize=12)
ax.set_title('H2分子能级对比 (惩罚系数: [3.0, 3.0])', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(states, fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8)

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

transitions = ['E1-E0', 'E2-E0', 'E2-E1']
qnn_exc = [qnn_ex1_final - qnn_gs_final, qnn_ex2_final - qnn_gs_final, qnn_ex2_final - qnn_ex1_final]
ffnn_exc = [ffnn_ex1_final - ffnn_gs_final, ffnn_ex2_final - ffnn_gs_final, ffnn_ex2_final - ffnn_ex1_final]
fci_exc = [E_fcis[1] - E_fcis[0], E_fcis[2] - E_fcis[0], E_fcis[2] - E_fcis[1]]

x = np.arange(len(transitions))
width = 0.25

rects1 = ax.bar(x - width, qnn_exc, width, label='QNN', color='blue', alpha=0.7)
rects2 = ax.bar(x, ffnn_exc, width, label='FFNN', color='red', alpha=0.7)
rects3 = ax.bar(x + width, fci_exc, width, label='FCI', color='green', alpha=0.7)

ax.set_ylabel('Excitation Energy (Ha)', fontsize=12)
ax.set_title('H2分子激发能对比', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(transitions, fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

plt.tight_layout()
plt.show()

## 8. 总结

本notebook对比了QNN和FFNN两种方法在H2分子激发态VMC计算中的表现：

### 计算设置：
- 第一激发态惩罚系数: [3.0]
- 第二激发态惩罚系数: [3.0, 3.0]

### 主要发现：
1. **基态计算**: 两种方法都能较好地逼近FCI基态能量
2. **第一激发态**: 使用惩罚系数3.0进行正交化约束
3. **第二激发态**: 使用惩罚系数[3.0, 3.0]同时与基态和第一激发态正交

### 方法对比：
- **QNN**: 使用量子电路进行特征提取，具有量子纠缠特性
- **FFNN**: 使用经典神经网络，计算效率更高

两种方法各有优势，选择取决于具体的应用需求和计算资源。